In [ ]:
import torch
from matplotlib import pyplot as plt

from gpsr.beams import NSF

In [ ]:
def estimate_entropy_knn(x: torch.Tensor, k: int = 1) -> torch.Tensor:    
    distances = torch.cdist(x, x, p=2)
    knn_distances, knn_indices = torch.topk(distances, k, largest=False, dim=1)
    epsilon = knn_distances[:, -1]
    entropy = torch.mean(torch.log(epsilon + 1.00e-12))  # ignores constant factors/terms 
    return entropy

In [ ]:
ndim = 2
nsamp = 2_000

model = NSF(ndim=2, cov_matrix=torch.eye(ndim))

with torch.no_grad():
    x = model.sample(50_000)
    fig, ax = plt.subplots(figsize=(3, 3))
    ax.hist2d(x[:, 0], x[:, 1], bins=100)
    plt.show()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for iteration in range(200):
    optimizer.zero_grad()

    x = model.sample(nsamp)

    loss = 0.0
    loss += torch.abs(torch.mean(x))
    loss += torch.abs(torch.std(x[:, 0]) - 1.0)
    loss += torch.abs(torch.std(x[:, 1]) - 1.0)
    loss -= estimate_entropy_knn(x, k=5)

    loss.backward()
    optimizer.step()
    
    if iteration % 25 == 0:
        print(iteration, loss, torch.mean(log_prob))

with torch.no_grad():
    x = model.sample(50_000)
    fig, ax = plt.subplots(figsize=(3, 3))
    ax.hist2d(x[:, 0], x[:, 1], bins=100, range=(2 * [(-5, 5)]))
    plt.show()

In [ ]:
model = NSF(ndim=2, cov_matrix=torch.eye(ndim))

with torch.no_grad():
    x = model.sample(50_000)
    fig, ax = plt.subplots(figsize=(3, 3))
    ax.hist2d(x[:, 0], x[:, 1], bins=100)
    plt.show()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for iteration in range(300):
    optimizer.zero_grad()

    x, log_prob = model.sample_and_log_prob(nsamp)

    loss = 0.0
    loss += torch.abs(torch.mean(x))
    loss += torch.abs(torch.std(x[:, 0]) - 1.0)
    loss += torch.abs(torch.std(x[:, 1]) - 1.0)
    loss += torch.mean(log_prob)

    loss.backward()
    optimizer.step()
    
    if iteration % 25 == 0:
        print(iteration, loss, torch.mean(log_prob))

with torch.no_grad():
    x = model.sample(50_000)
    fig, ax = plt.subplots(figsize=(3, 3))
    ax.hist2d(x[:, 0], x[:, 1], bins=100, range=(2 * [(-5, 5)]))
    plt.show()